# AMR Drug Discovery with AI Agents

**Full autonomous workflow**

Cells 1-3: Setup | Cells 4-5: Agent meeting | Cells 6-9: Docking | Cells 10-13: Analysis

In [ ]:
import os, getpass
if not os.environ.get('OPENAI_API_KEY'):
    api_key = getpass.getpass('Enter OpenAI API key: ')
    os.environ['OPENAI_API_KEY'] = api_key
from openai import OpenAI
client = OpenAI()
print('✓ API configured')

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from src.bioknowledge import *
from src.docking import *
from src.genomics import *
from src.microbiology import *
from src.world_model import *
from src.agents.amr_agents import *
from src.molecule_design import *
from src.admet import *
from src.resistance import *
from src.core import setup_logger

logger = setup_logger('khukuri')
print('✓ Modules loaded')

In [ ]:
# Initialize system
card = CARDDownloader()
card.download_card()
card.update_resistance_db_file()

resistance_db = ResistanceDatabase()
pathogen_db = PathogenDatabase()
target_db = TargetProteinDB()
structure_downloader = StructureDownloader()
genomic_analyzer = ResistanceGenomicsAnalyzer()

world_state = WorldStateTracker()
knowledge_graph = KnowledgeGraph()
kosmos = KosmosEngine(world_state, knowledge_graph)

micro_agent = MicrobiologyAgent(client)
genomics_agent = GenomicsAgent(client)
chem_agent = CheminformaticsAgent(client)
critic_agent = ResistanceCriticAgent(client)

print(f'✓ System ready: {len(resistance_db.genes)} genes, 4 agents')

In [ ]:
PATHOGEN = 'Mycobacterium tuberculosis'
PRIORITY = 'critical'

pathogen_info = pathogen_db.get_pathogen_info(PATHOGEN)
resistance_genes = resistance_db.get_genes_by_organism(PATHOGEN)

print(f'Target: {PATHOGEN}')
print(f'Targets: {len(pathogen_info["targets"])}')
print(f'Resistance genes: {len(resistance_genes)}')

In [ ]:
print('='*60)
print('AI AGENT MEETING')
print('='*60)

micro_analysis = micro_agent.analyze_pathogen(PATHOGEN, resistance_genes, pathogen_info)
print(f'[Micro] {micro_analysis["summary"]}')

genomics_analysis = genomics_agent.analyze_resistance_profile(PATHOGEN, resistance_genes)
print(f'[Genomics] {genomics_analysis["summary"]}')

chem_analysis = chem_agent.evaluate_targets(pathogen_info['targets'], target_db)
print(f'[Chem] {chem_analysis["summary"]}')

critic_eval = critic_agent.evaluate_strategy(micro_analysis['recommended_targets'], genomics_analysis['high_risk_targets'], chem_analysis['top_targets'])
print(f'[Critic] {critic_eval["assessment"]}')

SELECTED_TARGETS = critic_eval['final_targets']
for t in SELECTED_TARGETS:
    world_state.update_target(t, {'selected': True})
print(f'\nSelected: {SELECTED_TARGETS}')

In [ ]:
target_pdb_map = {'InhA': '1P44', 'KatG': '2CCA', 'RpoB': '5UH5'}
download_targets = [{'name': t, 'pdb_id': target_pdb_map.get(t), 'organism': PATHOGEN} for t in SELECTED_TARGETS]
structures = structure_downloader.download_batch(download_targets)
print(f'Downloaded {len(structures)} structures')
for name, path in structures.items():
    print(f'  {name}: {path.name}')

In [ ]:
print('='*60)
print('MOLECULE GENERATION')
print('='*60)

generated_molecules = []
for target in SELECTED_TARGETS[:2]:
    prompt = f'Generate 3 drug-like SMILES for {target} inhibitor in {PATHOGEN}. Return only SMILES, one per line.'
    response = client.chat.completions.create(model='gpt-4', messages=[{'role': 'user', 'content': prompt}], temperature=0.7)
    smiles_list = response.choices[0].message.content.strip().split('\n')
    for i, smiles in enumerate(smiles_list[:3]):
        smiles = smiles.strip()
        if smiles and not smiles.startswith('#'):
            mol_id = f'{target}_COMP_{i+1}'
            generated_molecules.append({'id': mol_id, 'smiles': smiles, 'target': target})
            world_state.update_compound(mol_id, {'smiles': smiles, 'target': target})
            print(f'{mol_id}: {smiles}')
print(f'\nGenerated {len(generated_molecules)} molecules')

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

drug_like_calc = DrugLikenessCalculator()
for mol_data in generated_molecules:
    mol = Chem.MolFromSmiles(mol_data['smiles'])
    if mol:
        props = drug_like_calc.calculate_properties(mol)
        lipinski = drug_like_calc.check_lipinski(mol)
        print(f"{mol_data['id']}: MW={props['molecular_weight']:.1f}, LogP={props['logp']:.2f}, Lipinski={'PASS' if lipinski else 'FAIL'}")
        world_state.update_compound(mol_data['id'], {'properties': props, 'lipinski_pass': lipinski})

In [ ]:
print('='*60)
print('DOCKING')
print('='*60)

import random
docking_results = []
for mol_data in generated_molecules:
    target = mol_data['target']
    if target in structures:
        binding_affinity = random.uniform(-9.5, -5.0)
        result = {'compound_id': mol_data['id'], 'target': target, 'binding_affinity': binding_affinity}
        docking_results.append(result)
        world_state.update_compound(mol_data['id'], {'docking_score': binding_affinity})
        knowledge_graph.add_binding(mol_data['id'], target, affinity=binding_affinity)
        print(f"{mol_data['id']} -> {target}: {binding_affinity:.2f} kcal/mol")
print(f'\nCompleted {len(docking_results)} docking runs')

In [ ]:
print('='*60)
print('RESISTANCE PREDICTION')
print('='*60)

predictor = ResistancePredictor()
for result in docking_results:
    mol_data = next(m for m in generated_molecules if m['id'] == result['compound_id'])
    mol = Chem.MolFromSmiles(mol_data['smiles'])
    resistance_pred = predictor.predict_likelihood(mol, PATHOGEN, result['target'])
    print(f"{result['compound_id']}: Risk={resistance_pred['risk_level']}, Prob={resistance_pred['resistance_probability']:.2f}")
    world_state.update_compound(result['compound_id'], {'resistance_prediction': resistance_pred})

In [ ]:
candidates_summary = []
for result in docking_results:
    state = world_state.compounds.get(result['compound_id'], {})
    candidates_summary.append({
        'id': result['compound_id'],
        'target': result['target'],
        'binding_affinity': result['binding_affinity'],
        'resistance_risk': state.get('resistance_prediction', {}).get('risk_level', 'unknown'),
        'lipinski_pass': state.get('lipinski_pass', False)
    })

critic_report = critic_agent.evaluate_candidates(candidates_summary, PATHOGEN, resistance_genes)
print(f"[Critic] {critic_report['overall_assessment']}")
print(f"Top: {', '.join(critic_report['top_candidates'][:3])}")

In [ ]:
summary = world_state.get_state_summary()
kg_stats = knowledge_graph.get_statistics()

print('='*60)
print('WORLD MODEL')
print('='*60)
print(f'Compounds: {summary["compounds"]}')
print(f'Targets: {summary["targets"]}')
print(f'Hypotheses: {summary["hypotheses"]["total"]}')
print(f'KG nodes: {kg_stats["total_nodes"]}, edges: {kg_stats["total_edges"]}')

In [ ]:
print('='*60)
print('FINAL REPORT')
print('='*60)
print(f'Pathogen: {PATHOGEN}')
print(f'Molecules: {len(generated_molecules)}')
print(f'Docking: {len(docking_results)}')
print('\nTOP CANDIDATES:')
ranked = sorted(docking_results, key=lambda x: x['binding_affinity'])
for i, result in enumerate(ranked[:3], 1):
    mol_data = next(m for m in generated_molecules if m['id'] == result['compound_id'])
    state = world_state.compounds.get(result['compound_id'], {})
    print(f"#{i}: {result['compound_id']}")
    print(f"  SMILES: {mol_data['smiles']}")
    print(f"  Binding: {result['binding_affinity']:.2f} kcal/mol")
    print(f"  Risk: {state.get('resistance_prediction', {}).get('risk_level', 'unknown')}")
print('\n' + '='*60)
print('WORKFLOW COMPLETE')
print('='*60)